# Adapt saved weights without changing their vocabulary

The default path continues the small model from [train-gpt.ipynb](train-gpt.ipynb), using the bundled corpus and no network. You can select GPT-2 explicitly for a larger pretrained experiment.

**You will:** distinguish continuing weights from resuming optimizer state, preserve token IDs, use a smaller learning rate, and compare validation loss before and after adaptation. The architecture and teaching approach build on Andrej Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT).

This notebook starts a **new optimizer and schedule** from existing model weights. It is fine-tuning, not an exact continuation of a paused optimizer. The default reuses the same corpus to keep the workflow self-contained; choose related new text to study domain adaptation.


## Environment and reproducibility

Use the environment described in the [README](README.md); Python 3.13 is the recommended starting point. From the repository root, run:

```sh
python -m pip install -e ".[notebooks]"
python -m jupyter lab
```

Choose the Python kernel from that environment and run cells **in order**. Installation is a one-time terminal step, not part of the lesson. Paths below work on Windows, macOS, and Linux. `auto` selects an available accelerator or CPU, then a supported precision. A seed makes comparisons more useful, but exact results can differ across hardware and library versions.


In [ ]:
from pathlib import Path
import json
import os
import sys

import torch
import jadegpt

ROOT = Path.cwd().resolve()
if not (ROOT / "jadegpt.py").is_file():
    raise RuntimeError("Start JupyterLab from the jadeGPT repository root.")

SEED = 1337
DEVICE = jadegpt.resolve_device("auto")
DTYPE = jadegpt.resolve_dtype("auto", DEVICE)
# A few CPU threads often suit a tiny model better than every available core.
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | {DEVICE} | {DTYPE}")


## 1. Choose exactly one source

`SOURCE="checkpoint"` uses the completed scratch lesson. Changing it to `"gpt2"` explicitly enables a first-use Hugging Face download of the smallest pretrained GPT-2 plus its tokenizer. That option needs internet access and several GB of available memory during loading/training; it is much slower on CPU. The roughly 124-million-parameter GPT-2 is still far larger than our scratch model. Larger GPT-2 variants are intentionally not the default.

Keep `SOURCE="checkpoint"` for an offline lesson or smoke test. Choosing one source avoids accidentally loading a pretrained model and then immediately overwriting it with a checkpoint.


In [ ]:
SOURCE = "checkpoint"  # "checkpoint" or "gpt2"; GPT-2 opts into downloads.
PRETRAINED_MODEL = "gpt2"
BASE_RUN_PATH = ROOT / "artifacts" / "notebooks" / "scratch" / "run.json"
WORK_DIR = ROOT / "artifacts" / "notebooks" / "finetune"
DATA_DIR = WORK_DIR / "data"
INPUT_PATH = ROOT / "examples" / "tiny_corpus.txt"

original_metadata = None
if SOURCE == "checkpoint":
    if not BASE_RUN_PATH.is_file():
        raise FileNotFoundError("Complete train-gpt.ipynb first, or select an existing run.json.")
    base_run = json.loads(BASE_RUN_PATH.read_text(encoding="utf-8"))
    original_metadata = jadegpt.load_metadata(ROOT / base_run["meta_path"])
    model = jadegpt.resume_gpt(ROOT / base_run["checkpoint_path"], random_seed=SEED, device=DEVICE)
    USE_GPT2_ENCODING = original_metadata["encoding"] == "gpt2"
    if USE_GPT2_ENCODING:
        assert model.config.vocab_size >= original_metadata["actual_vocab_size"]
    else:
        assert model.config.vocab_size == original_metadata["vocab_size"]
elif SOURCE == "gpt2":
    model = jadegpt.init_gpt2(gpt2_model=PRETRAINED_MODEL, random_seed=SEED)
    USE_GPT2_ENCODING = True
else:
    raise ValueError("SOURCE must be 'checkpoint' or 'gpt2'.")

BLOCK_SIZE = min(64, model.config.block_size)
print(f"Source: {SOURCE} | training context: {BLOCK_SIZE} | vocabulary: {model.config.vocab_size}")


## 2. Prepare text with the existing tokenizer

A fine-tuning corpus often omits some old characters. Rebuilding an alphabet from only those characters would renumber IDs and scramble the checkpoint. We preserve the full original character alphabet and reject unknown characters. GPT-2 uses its fixed byte-pair tokenizer instead; a token may be part of a word, a whole word, punctuation, or whitespace.

Split the new text before batching and leave enough tokens in each part. The saved data and adapted weights go into a separate `finetune` folder so the original run remains available for comparison.


In [ ]:
text = jadegpt.open_dataset_file(INPUT_PATH)
train_text, val_text = jadegpt.split_dataset(text, split=0.9)
vocabulary_source = text
if not USE_GPT2_ENCODING:
    unknown = sorted(set(text) - set(original_metadata["stoi"]))
    if unknown:
        raise ValueError(f"New corpus contains characters outside the original vocabulary: {unknown!r}")
    # The exporter sorts this same alphabet, preserving all original IDs.
    vocabulary_source = "".join(original_metadata["stoi"])

stats = jadegpt.export_data_to_files(
    data=vocabulary_source, train_data=train_text, val_data=val_text,
    use_gpt2_encoding=USE_GPT2_ENCODING, data_dir=DATA_DIR,
    train_file_name="train.bin", val_file_name="val.bin", meta_file_name="meta.json",
)
metadata = jadegpt.load_metadata(DATA_DIR / "meta.json")
if not USE_GPT2_ENCODING:
    assert metadata["stoi"] == original_metadata["stoi"], "Token IDs must remain identical."
if USE_GPT2_ENCODING:
    assert model.config.vocab_size >= metadata["actual_vocab_size"]
else:
    assert metadata["vocab_size"] == model.config.vocab_size, "Model/tokenizer vocabulary mismatch."
model.tokenizer_metadata = metadata
train_data = jadegpt.load_data_file_to_memmap(DATA_DIR, "train.bin")
val_data = jadegpt.load_data_file_to_memmap(DATA_DIR, "val.bin")
assert min(len(train_data), len(val_data)) > BLOCK_SIZE, "Add more text or reduce the context window."
print(f"Training tokens: {len(train_data):,} | held-out tokens: {len(val_data):,}")


## 3. Establish a held-out baseline

Measure the same fixed validation batches before and after training. Reusing a seed for these two measurements reduces sampling noise; it does not make a tiny validation set representative of every possible use. We use full precision for this small comparison so the same evaluation path works on CPU and accelerators.


In [ ]:
from contextlib import nullcontext

model.to(DEVICE)
def measure_validation():
    torch.manual_seed(SEED + 1)
    return float(jadegpt.estimate_loss(
        model=model, eval_iters=5, ctx=nullcontext(),
        train_data=train_data, val_data=val_data,
        device=DEVICE, block_size=BLOCK_SIZE, batch_size=2,
    )["val"])

before_loss = measure_validation()
print(f"Before fine-tuning: {before_loss:.3f} nats / token")


## 4. Adapt gently

Start with a smaller learning rate than scratch training. A high rate on a small corpus can quickly damage useful pretrained behavior or overfit the new text. Thirty updates are a workflow demonstration. Increase the budget gradually, watch held-out loss, and keep a baseline checkpoint.

Training GPT-2 updates all weights here; this is not a parameter-efficient method such as LoRA. If memory is tight, reduce batch size first, then context length. A small batch still requires space for weights, gradients, and optimizer state.


In [ ]:
MAX_ITERS = 30
LEARNING_RATE = 3e-4 if SOURCE == "checkpoint" else 5e-5
torch.manual_seed(SEED)
result = jadegpt.train_gpt(
    model=model, dtype=DTYPE, device=DEVICE,
    train_data=train_data, val_data=val_data,
    block_size=BLOCK_SIZE, batch_size=2, max_iters=MAX_ITERS,
    weight_decay=0.1, learning_rate=LEARNING_RATE, beta1=0.9, beta2=0.99,
    warmup_iters=min(3, MAX_ITERS - 1), lr_decay_iters=MAX_ITERS,
    min_lr=LEARNING_RATE / 10, decay_lr=True,
    eval_interval=10, eval_iters=3, gradient_accumulation_steps=1,
    grad_clip=1.0, log_interval=10, only_save_on_finish=True,
    save_interval=MAX_ITERS, model_dir=WORK_DIR / "checkpoints", model_name="adapted-gpt",
)
after_loss = measure_validation()
print(f"Fixed held-out batches: {before_loss:.3f} → {after_loss:.3f} nats / token")


In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
fig, ax = plt.subplots(figsize=(8, 3.5))
for key, label in (("train_loss", "training"), ("val_loss", "validation")):
    rows = [row for row in history if row.get(key) is not None]
    if rows:
        ax.plot([row["step"] for row in rows], [row[key] for row in rows], marker="o", label=label)
ax.set(xlabel="Optimizer updates", ylabel="Cross-entropy (nats / token)", title="Learning to predict the next token")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print(f"Finished {result['iterations']} updates; best validation loss: {result['best_val_loss']:.3f}")


In [ ]:
# Keep the exact tokenizer alongside the checkpoint: token IDs have meaning.
manifest = {
    "checkpoint_path": Path(result["checkpoint_path"]).resolve().relative_to(ROOT).as_posix(),
    "meta_path": (DATA_DIR / "meta.json").relative_to(ROOT).as_posix(),
    "encoding": metadata["encoding"],
    "seed": SEED,
    "iterations": result["iterations"],
    "best_val_loss": result["best_val_loss"],
}
RUN_PATH = WORK_DIR / "run.json"
RUN_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"Saved run: {RUN_PATH.relative_to(ROOT)}")
print(f"Checkpoint: {manifest['checkpoint_path']}")


## 5. Sample and decide what improved

Keep the prompt, seed, and sampling settings fixed when comparing the base and adapted model. A lower held-out loss on this corpus measures fit to this corpus; it does not establish factual accuracy or general usefulness. For character models, the prompt must stay inside the original alphabet.


In [ ]:
torch.manual_seed(SEED)
output = jadegpt.generate_text(
    model=model, start="the ", use_gpt2_encoding=USE_GPT2_ENCODING,
    meta_dir=DATA_DIR, meta_file_name="meta.json",
    num_samples=1, max_new_tokens=120, temperature=0.8, top_k=20,
    device=DEVICE, dtype=DTYPE,
)
print(output)


## Exercises

1. Load `artifacts/notebooks/finetune/run.json` in [sample-gpt.ipynb](sample-gpt.ipynb) and compare it with the scratch run.
2. Try related new text using the same alphabet. Explain why a character absent from the new text must keep its old token ID.
3. Compare two learning rates from the same base checkpoint. Which improves held-out loss without increasing repetition?
4. For a GPT-2 experiment, keep a separate held-out corpus from the original domain. Does adaptation help one domain while hurting the other?

If training loss falls while validation loss rises, investigate overfitting, dataset duplication, and the train/validation split before making the model larger.
